# Cross-Encoder: base vs fine-tuned (3 ключевые метрики)

Сравнение `DiTy/cross-encoder-russian-msmarco` (zero-shot) и дообученного варианта (`models/final/cross-encoder`) на `golden_eval.parquet`.

Метрики:
1. **Spearman ρ** — ранговая корреляция предсказанного скора с gold. Главная метрика.
2. **Pearson r** — линейная корреляция / калибровка.
3. **MAE** — средний абсолютный промах. Для cross-encoder важен, т.к. модель явно регрессирует скор, а не только порядок.

In [1]:
# Поднимаемся к корню thesis/, чтобы относительные пути работали из подпапки
import os
from pathlib import Path
_p = Path.cwd()
while _p.name != 'thesis' and _p.parent != _p:
    _p = _p.parent
if _p.name == 'thesis':
    os.chdir(_p)
print('CWD:', Path.cwd())


CWD: c:\Users\Admin\Documents\диплом\thesis


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from scipy import stats
from datasets import load_dataset
from sentence_transformers import CrossEncoder
from IPython.display import display

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')


DEVICE: cpu


In [3]:
BASE_MODEL       = 'DiTy/cross-encoder-russian-msmarco'
FINETUNED_MODEL  = 'models/final/cross-encoder'
EVAL_PARQUET     = 'data/golden/golden_eval.parquet'
BATCH_SIZE       = 32


In [4]:
# Загружаем golden_eval.parquet
ds = load_dataset('parquet', data_files=EVAL_PARQUET, split='train')
descriptions = list(ds['product_desc'])
posts        = list(ds['post_text'])
gold_scores  = np.array(ds['score'], dtype=float)
pairs = list(zip(descriptions, posts))
print(f'Пар для оценки: {len(pairs):,}')
print(f'Gold: min={gold_scores.min():.3f}, max={gold_scores.max():.3f}, mean={gold_scores.mean():.3f}')


Пар для оценки: 1,598
Gold: min=0.000, max=1.000, mean=0.336


In [5]:
# Три ключевые метрики cross-encoder:
# 1) Spearman ρ — качество ранжирования (главная)
# 2) Pearson r  — линейная корреляция / калибровка
# 3) MAE        — абсолютный промах (cross-encoder учится регрессировать скор,
#                  поэтому калиброванный промах имеет смысл)
def evaluate_cross_encoder(model, pairs, gold):
    scores = model.predict(pairs, batch_size=BATCH_SIZE, show_progress_bar=False)
    scores = np.asarray(scores, dtype=float).flatten()
    spearman_r, _ = stats.spearmanr(gold, scores)
    pearson_r, _  = stats.pearsonr(gold, scores)
    mae = float(np.mean(np.abs(gold - scores)))
    return {
        'spearman': spearman_r,
        'pearson':  pearson_r,
        'mae':      mae,
        'scores':   scores,
    }


In [6]:
import time

print(f'Загрузка базовой модели: {BASE_MODEL}')
base_model = CrossEncoder(BASE_MODEL, device=DEVICE)

print(f'Загрузка дообученной модели: {FINETUNED_MODEL}')
ft_model = CrossEncoder(FINETUNED_MODEL, device=DEVICE)

t0 = time.time()
base_res = evaluate_cross_encoder(base_model, pairs, gold_scores)
print(f'Base: {time.time()-t0:.0f}с  Spearman={base_res["spearman"]:.4f}  '
      f'Pearson={base_res["pearson"]:.4f}  MAE={base_res["mae"]:.4f}')

t0 = time.time()
ft_res = evaluate_cross_encoder(ft_model, pairs, gold_scores)
print(f'Fine-tuned: {time.time()-t0:.0f}с  Spearman={ft_res["spearman"]:.4f}  '
      f'Pearson={ft_res["pearson"]:.4f}  MAE={ft_res["mae"]:.4f}')


Загрузка базовой модели: DiTy/cross-encoder-russian-msmarco
Загрузка дообученной модели: models/final/cross-encoder


The tokenizer you are loading from 'models/final/cross-encoder' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Base: 239с  Spearman=0.2368  Pearson=0.3392  MAE=0.2973
Fine-tuned: 261с  Spearman=0.2882  Pearson=0.4418  MAE=0.2829


## Сводная таблица метрик

In [7]:
# Сводная таблица: Spearman, Pearson, MAE
df = pd.DataFrame([
    {
        'Модель':     'Cross-encoder base',
        'Spearman ρ': round(base_res['spearman'], 4),
        'Pearson r':  round(base_res['pearson'],  4),
        'MAE':        round(base_res['mae'],      4),
    },
    {
        'Модель':     'Cross-encoder fine-tuned',
        'Spearman ρ': round(ft_res['spearman'], 4),
        'Pearson r':  round(ft_res['pearson'],  4),
        'MAE':        round(ft_res['mae'],      4),
    },
])
print(f'Метрики cross-encoder на golden_eval.parquet ({len(pairs)} пар)')
display(df.style
        .background_gradient(cmap='YlGn', subset=['Spearman ρ', 'Pearson r'])
        .background_gradient(cmap='YlOrRd_r', subset=['MAE']))


Метрики cross-encoder на golden_eval.parquet (1598 пар)


,Модель,Spearman ρ,Pearson r,MAE
0,Cross-encoder base,0.236800,0.339200,0.297300
1,Cross-encoder fine-tuned,0.288200,0.441800,0.282900


## Дельта после дообучения

In [8]:
# Дельта fine-tuned − base: смотрим, что изменилось
delta = {
    'Δ Spearman ρ': round(ft_res['spearman'] - base_res['spearman'], 4),
    'Δ Pearson r':  round(ft_res['pearson']  - base_res['pearson'],  4),
    'Δ MAE':        round(ft_res['mae']      - base_res['mae'],      4),
}
print('Изменения после дообучения (положительное Δ Spearman/Pearson — улучшение; отрицательное Δ MAE — улучшение):')
for k, v in delta.items():
    arrow = '↑' if v > 0 else ('↓' if v < 0 else '=')
    print(f'  {k}: {v:+.4f}  {arrow}')


Изменения после дообучения (положительное Δ Spearman/Pearson — улучшение; отрицательное Δ MAE — улучшение):
  Δ Spearman ρ: +0.0513  ↑
  Δ Pearson r: +0.1026  ↑
  Δ MAE: -0.0144  ↓
